<a href="https://www.kaggle.com/code/kunrittyhe/nvidia-nemotron-submission-demo?scriptVersionId=308064534" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [3]:
SEED = 123

Code below is to fix permission error with ptxas-blackwell. Credit: [Aman Atar](https://www.kaggle.com/code/amanatar/nemotron-3-nano-blackwell-optimized-reasoning)

In [4]:
import os
import shutil, stat

# ptxas-blackwell fix
src = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell"
dst = "/tmp/ptxas-blackwell"
if os.path.exists(src):
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

    import triton.backends.nvidia as nv_backend
    src_bin = os.path.join(os.path.dirname(nv_backend.__file__), "bin")
    dst_bin = "/tmp/triton_nvidia_bin"
    shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

    nv_backend.__file__ = os.path.join(dst_bin, "..", "__init__.py")
    os.environ["TRITON_PTXAS_PATH"]           = dst
    os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = dst

    try:
        import triton.backends.nvidia.compiler as nv_compiler
        def _safe_get_ptxas_version(*args, **kwargs):
            return (12, 9, 0)
        nv_compiler.get_ptxas_version = _safe_get_ptxas_version
        print("Triton ptxas fix applied ✓")
    except Exception as e:
        print(f"Triton ptxas partial fix (non-fatal): {e}")
else:
    print("ptxas-blackwell not found — continuing without fix")

ptxas-blackwell not found — continuing without fix


Code to load model + LoRa below is forked from [Ryan Holbrook](https://www.kaggle.com/code/ryanholbrook/nvidia-nemotron-submission-demo). Roughly 7 minutes to run:

In [5]:
import site

cutlass_pkg_path = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/"
site.addsitedir(cutlass_pkg_path)

import kagglehub
import mamba_ssm
import torch
from peft import LoraConfig, get_peft_model, get_peft_model_state_dict, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer

# Configuration
MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
OUTPUT_DIR = "/kaggle/working"
LORA_RANK = 32  # Can be set to a maximum of 32

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.bfloat16
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
print("Model loaded successfully.")

# Initialize LoRA Adapter
print(f"Initializing LoRA adapter with rank={LORA_RANK}...")
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=16,
    target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


/opt/miniconda3/envs/nemotron-lora/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'mamba_ssm'

In [ ]:
import polars as pl

def load_train_test_split(csv_path, train_ratio=0.9, seed=123):
    df = pl.read_csv(csv_path)
    df = df.sample(fraction=1.0, seed=seed)
    
    split = int(len(df) * train_ratio)
    train = df[:split]
    eval = df[split:]
    
    print(f'Total: n={df.shape[0]}')
    print(f'Train: n={train.shape[0]}')
    print(f'Eval:  n={eval.shape[0]}')
    
    return train, eval

train, eval = load_train_test_split('/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv', seed=SEED)

Total: n=9500
Train: n=8550
Eval:  n=950


For now: Only use training sample beacuse of training speed + memory overheads

In [ ]:
#Sample
#train_sample = train.sample(5000, seed=SEED)
train_sample = train

#Format text with prompt and answer
train_texts = []
for row in train_sample.iter_rows(named=True):
    text = row['prompt'] + "\n" + str(row['answer'])
    train_texts.append(text)

Wrap training data to use with HuggingFace's Trainer:

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from torch.utils.data import Dataset

#Dataset Wrapper for Pytorch Trainer
class PromptDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=512):
        self.encodings = tokenizer(
            texts, truncation=True, padding="max_length",
            max_length=max_length, return_tensors="pt"
        )
    def __len__(self):
        return len(self.encodings["input_ids"])
    def __getitem__(self, idx):
        return {key: val[idx] for key, val in self.encodings.items()}

#Fix for pad_token error
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    
dataset = PromptDataset(train_texts, tokenizer)


The model + LoRa takes up roughly 63GB. AdamW optimizer takes up roughly another 20GB

In [ ]:
training_args = TrainingArguments(
    output_dir="./output",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    bf16=True, #IMPORTANT for Nemotron               
    logging_steps=10,
    save_strategy="no", #Change to epoch for long runs for failsafe
    dataloader_num_workers=4, #More GPU storage intensive to use 4. Use 2 for lighter load
    dataloader_pin_memory=True,
    # gradient_checkpointing=True, #Turn on to reduce GPU storage but increase training time
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

trainer.train()

Step,Training Loss
10,10.028403
20,5.883860
30,4.787471
40,4.927076
50,4.629918
60,4.404089
70,4.148119
80,4.269958
90,4.276015
100,4.752613


TrainOutput(global_step=3207, training_loss=3.8165672327225404, metrics={'train_runtime': 30349.5362, 'train_samples_per_second': 0.845, 'train_steps_per_second': 0.106, 'total_flos': 2.5298306805694464e+18, 'train_loss': 3.8165672327225404, 'epoch': 3.0})

In [ ]:
# Save Adapter
print(f"Saving adapter to {OUTPUT_DIR}...")
model.save_pretrained(OUTPUT_DIR)

Saving adapter to /kaggle/working...


In [ ]:
#For competition submission
import subprocess

subprocess.run("zip -m submission.zip *", shell=True, check=True)

  adding: README.md (deflated 66%)
  adding: __notebook__.ipynb (deflated 85%)
  adding: adapter_config.json (deflated 55%)
  adding: adapter_model.safetensors (deflated 9%)
  adding: output/ (stored 0%)


CompletedProcess(args='zip -m submission.zip *', returncode=0)